In [ ]:
!pip install pyspark pandas pyarrow -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("1. Khoi tao Spark Session (Toi uu RAM)...")
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Spark_Data_Prep_Final_Val") \
    .config("spark.driver.memory", "10g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()

BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_TRANS = BASE_PATH + "processed_v2/cleaned_transactions.parquet"
INPUT_CUST = BASE_PATH + "processed_v2/customers_processed.parquet"
INPUT_ART = BASE_PATH + "processed_v2/articles_processed.parquet"
MASTER_CAND_DIR = BASE_PATH + "outputs_v2/master/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. Khoi tao Spark Session (Toi uu RAM)...


In [ ]:
print("2. Doc du lieu tu he thong...")
train_master = spark.read.parquet(MASTER_CAND_DIR + "train_master_candidates.parquet")
test_master = spark.read.parquet(MASTER_CAND_DIR + "test_master_candidates.parquet")
transactions = spark.read.parquet(INPUT_TRANS)
customers = spark.read.parquet(INPUT_CUST)
articles = spark.read.parquet(INPUT_ART)

print("3. Gan nhan va Downsampling (Ty le 10:1) cho tap Train...")
max_date = transactions.select(F.max("t_dat_date")).collect()[0][0]
test_start = max_date - datetime.timedelta(days=7)
val_start = test_start - datetime.timedelta(days=7)

# Tao Ground Truth cho tap Train
target_w7 = transactions.filter((F.col("t_dat_date") >= val_start) & (F.col("t_dat_date") < test_start)) \
    .select("customer_id", "article_id") \
    .dropDuplicates() \
    .withColumn("label", F.lit(1))

train_labeled = train_master.join(F.broadcast(target_w7), ["customer_id", "article_id"], "left").fillna({"label": 0})

positives = train_labeled.filter(F.col("label") == 1)
negatives = train_labeled.filter(F.col("label") == 0)

pos_count = positives.count()
neg_count = negatives.count()
fraction = min(1.0, (pos_count * 10) / neg_count if neg_count > 0 else 1.0)

negatives_sampled = negatives.sample(withReplacement=False, fraction=fraction, seed=42)
x_train_base = positives.unionByName(negatives_sampled)
x_test_base = test_master

print("Da chuan bi xong Tap Train va Test co ban (Da bao gom diem als_score va itemcf_score).")

2. Doc du lieu tu he thong...
3. Gan nhan va Downsampling (Ty le 10:1) cho tap Train...
Da chuan bi xong Tap Train va Test co ban (Da bao gom diem als_score va itemcf_score).


In [ ]:
print("4. Khoi tao Ham tinh toan Dac trung (Feature Engineering)...")

def calculate_features_for_window(base_df, end_date, window_days=42):
    start_date = end_date - datetime.timedelta(days=window_days)
    hist_trans = transactions.filter((F.col("t_dat_date") >= start_date) & (F.col("t_dat_date") < end_date))

    # --- TẠO NHÓM TUỔI ---
    customers_meta = customers.select("customer_id", "age") \
        .withColumn("age_group",
            F.when(F.col("age") < 25, "<25")
             .when((F.col("age") >= 25) & (F.col("age") <= 35), "25-35")
             .when((F.col("age") >= 36) & (F.col("age") <= 45), "36-45")
             .when((F.col("age") >= 46) & (F.col("age") <= 55), "46-55")
             .otherwise(">55")
        )

    # --- TÍNH NĂNG CƠ BẢN ---
    item_features = hist_trans.groupBy("article_id").agg(
        F.count("customer_id").alias("item_total_sales"),
        F.avg("price").alias("item_avg_price")
    )

    user_features = hist_trans.groupBy("customer_id").agg(
        F.count("article_id").alias("user_total_purchases"),
        F.avg("price").alias("user_avg_budget")
    )

    user_item_interaction = hist_trans.groupBy("customer_id", "article_id").agg(
        F.count("t_dat_date").alias("user_item_buy_count")
    )

    # --- RECENCY ---
    user_recency = hist_trans.groupBy("customer_id").agg(
        F.max("t_dat_date").alias("last_purchase_date")
    ).withColumn(
        "days_since_last_purchase",
        F.datediff(F.lit(end_date), F.col("last_purchase_date"))
    ).select("customer_id", "days_since_last_purchase")

    item_specific_recency = hist_trans.groupBy("customer_id", "article_id").agg(
        F.max("t_dat_date").alias("last_bought_this_item")
    ).withColumn(
        "days_since_bought_THIS_item",
        F.datediff(F.lit(end_date), F.col("last_bought_this_item"))
    ).select("customer_id", "article_id", "days_since_bought_THIS_item")

    # --- XU HƯỚNG (3D, 7D, 14D) ---
    item_trend_3d = hist_trans.filter(F.col("t_dat_date") >= end_date - datetime.timedelta(days=3)) \
        .groupBy("article_id").agg(F.count("customer_id").alias("item_sales_last_3d"))

    item_trend_7d = hist_trans.filter(F.col("t_dat_date") >= end_date - datetime.timedelta(days=7)) \
        .groupBy("article_id").agg(F.count("customer_id").alias("item_sales_last_7d"))

    item_trend_14d = hist_trans.filter(F.col("t_dat_date") >= end_date - datetime.timedelta(days=14)) \
        .groupBy("article_id").agg(F.count("customer_id").alias("item_sales_last_14d"))

    # --- SỞ THÍCH DANH MỤC VÀ TUỔI ---
    hist_with_type = hist_trans.join(F.broadcast(articles.select("article_id", "product_type_name")), "article_id", "inner")
    user_type_features = hist_with_type.groupBy("customer_id", "product_type_name").agg(
        F.count("*").alias("user_type_buy_count")
    )

    hist_with_age = hist_trans.join(F.broadcast(customers.select("customer_id", "age")), "customer_id", "inner")
    item_age_features = hist_with_age.groupBy("article_id").agg(
        F.avg("age").alias("item_avg_age")
    )

    hist_with_age_group = hist_trans.join(F.broadcast(customers_meta.select("customer_id", "age_group")), "customer_id", "inner")
    age_group_item_sales = hist_with_age_group.groupBy("age_group", "article_id").agg(
        F.count("*").alias("age_group_item_sales")
    )

    articles_meta = articles.select("article_id", "product_type_name", "colour_group_name")

    # --- GHÉP NỐI TOÀN BỘ ---
    df = base_df.join(F.broadcast(customers_meta), "customer_id", "left") \
                .join(F.broadcast(articles_meta), "article_id", "left") \
                .join(F.broadcast(item_features), "article_id", "left") \
                .join(F.broadcast(user_features), "customer_id", "left") \
                .join(F.broadcast(user_recency), "customer_id", "left") \
                .join(user_item_interaction, ["customer_id", "article_id"], "left") \
                .join(item_specific_recency, ["customer_id", "article_id"], "left") \
                .join(F.broadcast(item_trend_3d), "article_id", "left") \
                .join(F.broadcast(item_trend_7d), "article_id", "left") \
                .join(F.broadcast(item_trend_14d), "article_id", "left") \
                .join(F.broadcast(item_age_features), "article_id", "left") \
                .join(F.broadcast(user_type_features), ["customer_id", "product_type_name"], "left") \
                .join(F.broadcast(age_group_item_sales), ["age_group", "article_id"], "left")

    # --- ĐIỀN GIÁ TRỊ MẶC ĐỊNH (Bao gom ca diem so) ---
    df = df.fillna({
        "item_total_sales": 0, "item_avg_price": 0.02,
        "user_total_purchases": 0, "user_avg_budget": 0.02,
        "user_item_buy_count": 0, "days_since_last_purchase": 999,
        "days_since_bought_THIS_item": 999,
        "item_sales_last_3d": 0, "item_sales_last_7d": 0, "item_sales_last_14d": 0,
        "age_group_item_sales": 0,
        "age": 25,
        "product_type_name": "Unknown", "colour_group_name": "Unknown",
        "user_type_buy_count": 0, "item_avg_age": 25,
        "als_score": 0.0,        # [MỚI] Bao ve diem ALS
        "itemcf_score": 0.0      # [MỚI] Bao ve diem ItemCF
    })

    # --- CÁC BIẾN NÂNG CAO & CỜ ---
    df = df.withColumn("price_diff", F.abs(F.col("item_avg_price") - F.col("user_avg_budget")))
    df = df.withColumn("age_diff", F.abs(F.col("age") - F.col("item_avg_age")))
    df = df.withColumn("trend_velocity", F.col("item_sales_last_7d") / (F.col("item_sales_last_14d") + 1.0))

    df = df.withColumn("from_als", F.when(F.array_contains(F.col("sources"), "als"), 1).otherwise(0)) \
           .withColumn("from_itemcf", F.when(F.array_contains(F.col("sources"), "itemcf"), 1).otherwise(0))

    # Xoa cac cot trung gian de tranh loi PyArrow
    df = df.drop("sources", "age_group")

    return df

4. Khoi tao Ham tinh toan Dac trung (Feature Engineering)...


In [ ]:
print("5. Dang tinh dac trung cho tap Train (Moc: cuoi Tuan 6)...")
train_enriched = calculate_features_for_window(x_train_base, end_date=val_start)
train_out_path = MASTER_CAND_DIR + "train_enriched_temp.parquet"
train_enriched.write.mode("overwrite").parquet(train_out_path)

print("6. Dang tinh dac trung cho tap Test (Moc: cuoi Tuan 7)...")
test_enriched = calculate_features_for_window(x_test_base, end_date=test_start)
test_out_path = MASTER_CAND_DIR + "test_enriched_temp.parquet"
test_enriched.write.mode("overwrite").parquet(test_out_path)

print("Hoan tat qua trinh tien xu ly! Du lieu da san sang de day vao LightGBM.")

5. Dang tinh dac trung cho tap Train (Moc: cuoi Tuan 6)...
6. Dang tinh dac trung cho tap Test (Moc: cuoi Tuan 7)...
Hoan tat qua trinh tien xu ly! Du lieu da san sang de day vao LightGBM.
